<a href="https://colab.research.google.com/github/Newbluewood/ML-SentimentModelForReviews/blob/main/notebooks/product_category_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📥 Učitavanje i pregled skupa podataka

Pre analize učitavamo skup i gledamo strukturu — bez ikakvih izmena.

U ovom koraku:
- učitavamo CSV sa GitHub-a
- proveravamo broj redova i kolona
- prikazujemo prvih nekoliko redova

**Cilj projekta:** na osnovu **Product Title** predvideti **Category Label**.


In [ ]:
import pandas as pd

url = (
    "https://raw.githubusercontent.com/Newbluewood/"
    "ML-SentimentModelForReviews/main/data/IMLP4_TASK_03-products.csv"
)
df = pd.read_csv(url)
df.columns = df.columns.str.strip()


In [ ]:
print("Oblik skupa (redovi, kolone):", df.shape)
display(df.head())


**Zaključak:** Skup je učitan. Sledeći korak — tipovi kolona i broj popunjenih vrednosti.


In [ ]:
df.info()


## 🔍 Provera nedostajućih vrednosti

Nedostajući podaci kvare treniranje. Ovde samo **dijagnostikujemo** — još ne brišemo.


In [ ]:
print("Nedostajuće vrednosti po kolonama:")
print(df.isna().sum())


**Šta gledamo:** koje kolone imaju praznine i koliko ih ima.

Posebno su nam bitni **Product Title** i **Category Label** — bez njih red ne može u model.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, cmap="YlOrRd")
plt.title("Heatmap nedostajućih vrednosti")
plt.show()


**Zaključak:** Vizuelno vidimo gde su rupe. Pre čišćenja pregledamo raspodelu kategorija i ključne kolone.


## 📊 Analiza raspodele kategorija

Pregledamo ciljnu promenljivu na **sirovom** skupu — kao sentiment analiza u ITA-ML.


In [ ]:
raspodela_kategorija = df["Category Label"].value_counts()
print("Broj različitih kategorija:", df["Category Label"].nunique())
print(raspodela_kategorija)


**Šta tražimo:**
- da li su klase neuravnotežene
- da li postoje dupli nazivi iste kategorije (npr. `fridge` i `Fridges`)

Ako vidimo nekonzistentnosti, ispravićemo ih posle `dropna()`.


In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(
    data=df, y="Category Label",
    order=raspodela_kategorija.index,
    hue="Category Label", legend=False
)
plt.title("Raspodela proizvoda po kategorijama")
plt.tight_layout()
plt.show()


## 📝 Istraživanje kolone `Product Title`

Glavni ulaz modela. Pre čišćenja proveravamo kako je kolona zapisana.


In [ ]:
print("Tip kolone:", df["Product Title"].dtype)
print("\nPrvih 10 naslova:")
print(df["Product Title"].head(10).to_string(index=False))


In [ ]:
duzina_raw = df["Product Title"].astype(str).str.len()
print("Statistika dužine naslova (znakovi):")
print(duzina_raw.describe())
print("\nRedova bez naslova:", df["Product Title"].isna().sum())


**Zaključak:** Naslovi su tekst. Dužina varira po proizvodu — kasnije ćemo od toga napraviti karakteristiku.


## 💸 Istraživanje numeričkih kolona

Pregledamo `Merchant Rating` i `Number_of_Views` pre čišćenja — da znamo tipove i opseg vrednosti.


In [ ]:
print("Merchant Rating")
print("  tip:", df["Merchant Rating"].dtype)
print(df["Merchant Rating"].describe())
print("\nNumber_of_Views")
print("  tip:", df["Number_of_Views"].dtype)
print(df["Number_of_Views"].describe())


**Zaključak:** Obe kolone su numeričke. Zadržavamo ih do `dropna()`; detaljnu analizu korelacije sa kategorijom (kvartili + boxplot) radimo posle standardizacije kategorija.


## 🧹 Uklanjanje nedostajućih vrednosti

Sada uklanjamo **sve redove** gde bilo koja kolona ima `NaN` — isto kao `df.dropna()` u ITA-ML.

**Zašto?** Dobijamo potpuno popunjen skup bez imputacije.


In [ ]:
prethodni_broj = len(df)
df = df.dropna()
print("Pre:", prethodni_broj, "redova")
print("Posle:", df.shape[0], "redova")
print("Uklonjeno:", prethodni_broj - df.shape[0])


In [ ]:
print("Nedostajuće posle dropna:")
print(df.isna().sum())


**Zaključak:** Svaka ćelija ima vrednost. Sledeće — usklađivanje naziva kategorija.


## ✅ Standardizacija `Category Label`

Ispravljamo nekonzistentne nazive i postavljamo tip `category`.


In [ ]:
print("PRE standardizacije:")
print(df["Category Label"].value_counts())


In [ ]:
ispravka_kategorija = {
    "fridge": "Fridges",
    "CPU": "CPUs",
    "Mobile Phone": "Mobile Phones",
}
df["Category Label"] = (
    df["Category Label"].astype(str).str.strip().replace(ispravka_kategorija)
)
df["Category Label"] = df["Category Label"].astype("category")
print("POSLE standardizacije:")
print(df["Category Label"].value_counts())


**Zaključak:** Ostaje 10 čistih kategorija. Sledeće — čišćenje naslova.


## ✅ Uklanjanje razmaka u `Product Title`

Prvi korak čišćenja teksta — `strip()` sa početka i kraja.


In [ ]:
print("PRE:")
print(df["Product Title"].head(3).to_string(index=False))
df["Product Title"] = df["Product Title"].astype(str).str.strip()
print("\nPOSLE:")
print(df["Product Title"].head(3).to_string(index=False))


## ✂️ Uklanjanje suvišnih kolona

Ostavljamo samo `Product Title` i `Category Label` za model iz naslova.


In [ ]:
df = df.drop(columns=[
    "product ID", "Merchant ID", "_Product Code",
    "Number_of_Views", "Merchant Rating", "Listing Date"
])
print("Kolone:", df.columns.tolist())
print("Oblik:", df.shape)


**Zaključak:** Radni skup je spreman za feature engineering.


## 🧮 Kreiranje karakteristika iz naslova

Kao `review_length` u ITA-ML — brojčani signali iz teksta.

Prvo kreiramo karakteristike koje zavise od **velikih slova**, pa tek onda radimo lowercase.


In [ ]:
df["broj_reci"] = df["Product Title"].str.split().str.len()
df["duzina_naslova"] = df["Product Title"].str.len()
print(df[["broj_reci", "duzina_naslova"]].describe())


In [ ]:
df["ima_brojeve"] = df["Product Title"].str.contains(r"\d", regex=True).astype(int)
df["ima_velika_slova"] = df["Product Title"].str.contains(r"[A-Z]", regex=True).astype(int)
df["max_duzina_reci"] = df["Product Title"].str.split().apply(
    lambda reci: max((len(r) for r in reci), default=0)
)
print("ima_brojeve — udeo sa ciframa:", df["ima_brojeve"].mean())
print("ima_velika_slova — udeo:", df["ima_velika_slova"].mean())


**Zaključak:** Karakteristike kreirane. Proveravamo da li se dužina naslova razlikuje po kategorijama.


In [ ]:
print(df.groupby("Category Label", observed=False)["duzina_naslova"].mean().sort_values(ascending=False))


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Category Label", y="duzina_naslova")
plt.title("Dužina naslova po kategoriji")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## ✅ Pretvaranje naslova u mala slova

Posle FE — `lower()` za TF-IDF, da `iPhone` i `iphone` budu isti token.


In [ ]:
df["Product Title"] = df["Product Title"].str.lower()
print(df["Product Title"].head(3).to_string(index=False))


**Zaključak:** Podaci su očišćeni i pripremljeni. Sledeće — treniranje modela.


### 📚 Priprema za modeliranje

Definišemo ulaze (`X`), cilj (`y`) i delimo skup na trening i test.


In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = [
    "broj_reci", "duzina_naslova", "ima_brojeve",
    "ima_velika_slova", "max_duzina_reci"
]
X = df[["Product Title"] + numeric_features]
y = df["Category Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Trening:", X_train.shape, "| Test:", X_test.shape)


**Zaključak:** 80% trening, 20% test, ista raspodela klasa (`stratify`). Sledeće — preprocessor i Pipeline.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer(
    transformers=[
        ("title", TfidfVectorizer(), "Product Title"),
        ("numeric", MinMaxScaler(), numeric_features),
    ]
)
print("Preprocessor definisan.")


### Treniranje modela — Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

pipe_lr = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
print("Tačnost:", round(accuracy_score(y_test, y_pred_lr), 4))
print(classification_report(y_test, y_pred_lr))


### Treniranje modela — Naive Bayes


In [ ]:
from sklearn.naive_bayes import MultinomialNB

pipe_nb = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", MultinomialNB()),
])
pipe_nb.fit(X_train, y_train)
y_pred_nb = pipe_nb.predict(X_test)
print("Tačnost:", round(accuracy_score(y_test, y_pred_nb), 4))
print(classification_report(y_test, y_pred_nb))


### Treniranje modela — Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipe_rf = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
])
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)
print("Tačnost:", round(accuracy_score(y_test, y_pred_rf), 4))
print(classification_report(y_test, y_pred_rf))


**Zaključak:** Uporedili smo tri modela. Biramo najbolji po tačnosti za matricu zabune.


In [ ]:
rezultati = {
    "Logistic Regression": (pipe_lr, y_pred_lr, accuracy_score(y_test, y_pred_lr)),
    "Naive Bayes": (pipe_nb, y_pred_nb, accuracy_score(y_test, y_pred_nb)),
    "Random Forest": (pipe_rf, y_pred_rf, accuracy_score(y_test, y_pred_rf)),
}
najbolji_naziv = max(rezultati, key=lambda k: rezultati[k][2])
najbolji_pipeline, najbolja_predikcija, najbolja_tacnost = rezultati[najbolji_naziv]
print(f"Najbolji model: {najbolji_naziv} (tačnost {najbolja_tacnost:.4f})")


## 📉 Matrica zabune

Pokazuje koje kategorije se najčešće mešaju.


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, najbolja_predikcija, labels=najbolji_pipeline.classes_)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Matrica zabune — {najbolji_naziv} ({najbolja_tacnost:.2%})")
plt.xlabel("Predviđeno")
plt.ylabel("Stvarno")
plt.tight_layout()
plt.show()
